# Customer Data Engineering Pipeline — PySpark Analytics

## Overview

This notebook demonstrates local PySpark processing as the analytical layer of a customer data engineering pipeline.

The upstream customer data is maintained in SQL Server through an incremental ETL process. Active customer records are exported and processed locally with PySpark.

## Pipeline

SQL Server
→ Active customer dataset
→ CSV ingestion
→ PySpark DataFrame
→ Transformations and aggregations
→ Parquet storage
→ Partitioned customer dataset
→ Analytical metrics
→ Final Parquet output

## Objectives

- Demonstrate PySpark DataFrame operations
- Apply transformations and aggregations
- Demonstrate lazy evaluation and execution plans
- Use Parquet as a columnar storage format
- Demonstrate partitioning and partition pruning
- Perform basic data-quality validation
- Produce a reusable analytical dataset

In [2]:
!pip install pyspark

## 1. Spark Setup

Initialize a local Spark session for PySpark processing.

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (SparkSession.builder.appName("CustomerDataEngineeringProject").getOrCreate())

In [5]:
spark

## 2. Data Ingestion

Load the active customer dataset exported from SQL Server into a Spark DataFrame and inspect its inferred schema.

In [6]:
from google.colab import files

uploaded = files.upload()

Saving customers_active.csv to customers_active.csv


In [7]:
customers_spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("customers_active.csv")
)

In [8]:
customers_spark_df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- Gender: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- TotalChildren: integer (nullable = true)
 |-- HouseOwnerFlag: integer (nullable = true)
 |-- NumberCarsOwned: integer (nullable = true)
 |-- Education: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Continent: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- YearlyIncome: double (nullable = true)
 |-- DateFirstPurchase: date (nullable = true)



In [9]:
customers_spark_df.show(5)

+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------+---------------+-----------+------------+-----------------+
|CustomerID|     CustomerName| BirthDate|Gender|MaritalStatus|TotalChildren|HouseOwnerFlag|NumberCarsOwned|Education|  Occupation|Continent|  Country|          State|       City|YearlyIncome|DateFirstPurchase|
+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------+---------------+-----------+------------+-----------------+
|     11000|         Jon Yang|1971-10-06|     M|            M|            2|             1|              0|Bachelors|Professional|  Pacific|Australia|     Queensland|Rockhampton|     90000.0|       2011-01-19|
|     11001|     Eugene Huang|1976-05-10|     M|            S|            3|             0|              1|Bachelors|Professional|  Pacific|Australia|       Vic

In [10]:
customers_df = customers_spark_df

print(f"Rows: {customers_df.count()}")
print(f"Columns: {len(customers_df.columns)}")

customers_df.show(5)

Rows: 18484
Columns: 16
+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------+---------------+-----------+------------+-----------------+
|CustomerID|     CustomerName| BirthDate|Gender|MaritalStatus|TotalChildren|HouseOwnerFlag|NumberCarsOwned|Education|  Occupation|Continent|  Country|          State|       City|YearlyIncome|DateFirstPurchase|
+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------+---------------+-----------+------------+-----------------+
|     11000|         Jon Yang|1971-10-06|     M|            M|            2|             1|              0|Bachelors|Professional|  Pacific|Australia|     Queensland|Rockhampton|     90000.0|       2011-01-19|
|     11001|     Eugene Huang|1976-05-10|     M|            S|            3|             0|              1|Bachelors|Professional|  Paci

## 3. Customer Analytics

Perform analytical transformations on the active customer dataset using the PySpark DataFrame API.

The analysis includes customer counts and average yearly income by country.

In [11]:
from pyspark.sql.functions import avg, count, round

country_analysis_df = (
    customers_df
    .groupBy('Country')
    .agg(count('*').alias('CustomerCount'), round(avg('YearlyIncome'), 2).alias('AverageYearlyIncome')).orderBy('CustomerCount', ascending = False)
    )

country_analysis_df.show(20, truncate = False)

+--------------+-------------+-------------------+
|Country       |CustomerCount|AverageYearlyIncome|
+--------------+-------------+-------------------+
|United States |7819         |63616.83           |
|Australia     |3591         |64338.62           |
|United Kingdom|1913         |52169.37           |
|France        |1810         |35762.43           |
|Germany       |1780         |42943.82           |
|Canada        |1571         |57167.41           |
+--------------+-------------+-------------------+



### 3.1 Income Segmentation

Create an analytical customer segment based on yearly income.

In [12]:
from pyspark.sql.functions import when, col

customer_segments_df = (
    customers_df
    .withColumn(
        "IncomeSegment",
        when(col("YearlyIncome") >= 100000, "High Income")
        .otherwise("Standard Income")
    )
)

customer_segments_df.select(
    "CustomerID",
    "CustomerName",
    "YearlyIncome",
    "IncomeSegment"
).show(10)

+----------+-----------------+------------+---------------+
|CustomerID|     CustomerName|YearlyIncome|  IncomeSegment|
+----------+-----------------+------------+---------------+
|     11000|         Jon Yang|     90000.0|Standard Income|
|     11001|     Eugene Huang|     60000.0|Standard Income|
|     11002|     Ruben Torres|     60000.0|Standard Income|
|     11003|      Christy Zhu|     70000.0|Standard Income|
|     11004|Elizabeth Johnson|     80000.0|Standard Income|
|     11005|       Julio Ruiz|     70000.0|Standard Income|
|     11006|    Janet Alvarez|     70000.0|Standard Income|
|     11007|      Marco Mehta|     60000.0|Standard Income|
|     11008|      Rob Verhoff|     60000.0|Standard Income|
|     11009|  Shannon Carlson|     70000.0|Standard Income|
+----------+-----------------+------------+---------------+
only showing top 10 rows


In [13]:
from pyspark.sql.functions import count, avg, round

income_segment_analysis_df = (
    customer_segments_df
    .groupBy("IncomeSegment")
    .agg(
        count("*").alias("CustomerCount"),
        round(avg("YearlyIncome"), 2).alias("AverageYearlyIncome")
    )
    .orderBy("CustomerCount", ascending=False)
)

income_segment_analysis_df.show()

+---------------+-------------+-------------------+
|  IncomeSegment|CustomerCount|AverageYearlyIncome|
+---------------+-------------+-------------------+
|Standard Income|        16286|           48757.83|
|    High Income|         2198|          120641.49|
+---------------+-------------+-------------------+



### 3.2 Spark Execution Plan

Inspect the physical execution plan generated by Spark to demonstrate lazy evaluation and query planning.

In [14]:
customer_segments_df.explain()

== Physical Plan ==
*(1) Project [CustomerID#17, CustomerName#18, BirthDate#19, Gender#20, MaritalStatus#21, TotalChildren#22, HouseOwnerFlag#23, NumberCarsOwned#24, Education#25, Occupation#26, Continent#27, Country#28, State#29, City#30, YearlyIncome#31, DateFirstPurchase#32, CASE WHEN (YearlyIncome#31 >= 100000.0) THEN High Income ELSE Standard Income END AS IncomeSegment#226]
+- FileScan csv [CustomerID#17,CustomerName#18,BirthDate#19,Gender#20,MaritalStatus#21,TotalChildren#22,HouseOwnerFlag#23,NumberCarsOwned#24,Education#25,Occupation#26,Continent#27,Country#28,State#29,City#30,YearlyIncome#31,DateFirstPurchase#32] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/customers_active.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<CustomerID:int,CustomerName:string,BirthDate:date,Gender:string,MaritalStatus:string,Total...




## 4. Parquet Storage

Convert the ingested customer dataset from CSV to Parquet.

Parquet provides a columnar storage format that preserves data types and is well suited to analytical workloads.

In [15]:
customers_df.write \
    .mode("overwrite") \
    .parquet("customers_parquet")

In [16]:
customers_parquet_df = spark.read.parquet("customers_parquet")

customers_parquet_df.printSchema()
customers_parquet_df.show(5)

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- Gender: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- TotalChildren: integer (nullable = true)
 |-- HouseOwnerFlag: integer (nullable = true)
 |-- NumberCarsOwned: integer (nullable = true)
 |-- Education: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Continent: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- YearlyIncome: double (nullable = true)
 |-- DateFirstPurchase: date (nullable = true)

+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------+---------------+-----------+------------+-----------------+
|CustomerID|     CustomerName| BirthDate|Gender|MaritalStatus|TotalChildren|HouseOwnerFlag|NumberCarsOwned|

In [22]:
country_analysis_df = (
    customers_parquet_df
    .groupBy("Country")
    .agg(
        count("*").alias("CustomerCount"),
        round(avg("YearlyIncome"), 2).alias("AverageYearlyIncome")
    )
    .orderBy("CustomerCount", ascending=False)
)

In [25]:
country_analysis_df.show()

+--------------+-------------+-------------------+
|       Country|CustomerCount|AverageYearlyIncome|
+--------------+-------------+-------------------+
| United States|         7819|           63616.83|
|     Australia|         3591|           64338.62|
|United Kingdom|         1913|           52169.37|
|        France|         1810|           35762.43|
|       Germany|         1780|           42943.82|
|        Canada|         1571|           57167.41|
+--------------+-------------+-------------------+



### 4.1 Partitioned Customer Dataset

Store the customer-level dataset partitioned by country.

Partitioning organizes the physical data by a frequently filtered column and can reduce the amount of data Spark needs to scan for partition-specific queries.

In [23]:
customers_parquet_df.write \
    .mode("overwrite") \
    .partitionBy("Country") \
    .parquet("customers_by_country")

### 4.2 Partition Pruning

Query a single country from the partitioned dataset and inspect the physical execution plan to verify that Spark applies a partition filter.

In [26]:
australia_df = (
    spark.read
    .parquet("customers_by_country")
    .filter(col("Country") == "Australia")
)

australia_df.show(5)

+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------------+-----------+------------+-----------------+---------+
|CustomerID|     CustomerName| BirthDate|Gender|MaritalStatus|TotalChildren|HouseOwnerFlag|NumberCarsOwned|Education|  Occupation|Continent|          State|       City|YearlyIncome|DateFirstPurchase|  Country|
+----------+-----------------+----------+------+-------------+-------------+--------------+---------------+---------+------------+---------+---------------+-----------+------------+-----------------+---------+
|     11000|         Jon Yang|1971-10-06|     M|            M|            2|             1|              0|Bachelors|Professional|  Pacific|     Queensland|Rockhampton|     90000.0|       2011-01-19|Australia|
|     11001|     Eugene Huang|1976-05-10|     M|            S|            3|             0|              1|Bachelors|Professional|  Pacific|       Victoria|    

In [27]:
australia_df.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [CustomerID#479,CustomerName#480,BirthDate#481,Gender#482,MaritalStatus#483,TotalChildren#484,HouseOwnerFlag#485,NumberCarsOwned#486,Education#487,Occupation#488,Continent#489,State#490,City#491,YearlyIncome#492,DateFirstPurchase#493,Country#494] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/customers_by_country], PartitionFilters: [isnotnull(Country#494), (Country#494 = Australia)], PushedFilters: [], ReadSchema: struct<CustomerID:int,CustomerName:string,BirthDate:date,Gender:string,MaritalStatus:string,Total...




In [28]:
from pyspark.sql.functions import count, avg, round

country_metrics_df = (
    customers_parquet_df
    .groupBy("Country")
    .agg(
        count("*").alias("CustomerCount"),
        round(avg("YearlyIncome"), 2).alias("AverageYearlyIncome"),
        round(avg("TotalChildren"), 2).alias("AverageChildren"),
        round(avg("NumberCarsOwned"), 2).alias("AverageCarsOwned")
    )
    .orderBy("CustomerCount", ascending=False)
)

country_metrics_df.show(truncate=False)

+--------------+-------------+-------------------+---------------+----------------+
|Country       |CustomerCount|AverageYearlyIncome|AverageChildren|AverageCarsOwned|
+--------------+-------------+-------------------+---------------+----------------+
|United States |7819         |63616.83           |2.0            |1.52            |
|Australia     |3591         |64338.62           |1.63           |1.91            |
|United Kingdom|1913         |52169.37           |1.63           |1.22            |
|France        |1810         |35762.43           |1.74           |1.25            |
|Germany       |1780         |42943.82           |1.67           |1.21            |
|Canada        |1571         |57167.41           |2.12           |1.46            |
+--------------+-------------+-------------------+---------------+----------------+



In [29]:
country_metrics_df.write \
    .mode("overwrite") \
    .parquet("country_metrics")

## 5. Data Quality Validation

Validate the analytical output by checking the number of countries, required fields, and reconciliation of customer counts against the active customer population.

In [30]:
print(f"Number of countries: {country_metrics_df.count()}")

country_metrics_df.show(truncate=False)

Number of countries: 6
+--------------+-------------+-------------------+---------------+----------------+
|Country       |CustomerCount|AverageYearlyIncome|AverageChildren|AverageCarsOwned|
+--------------+-------------+-------------------+---------------+----------------+
|United States |7819         |63616.83           |2.0            |1.52            |
|Australia     |3591         |64338.62           |1.63           |1.91            |
|United Kingdom|1913         |52169.37           |1.63           |1.22            |
|France        |1810         |35762.43           |1.74           |1.25            |
|Germany       |1780         |42943.82           |1.67           |1.21            |
|Canada        |1571         |57167.41           |2.12           |1.46            |
+--------------+-------------+-------------------+---------------+----------------+



In [31]:
country_metrics_df.select(
    "Country",
    "CustomerCount",
    "AverageYearlyIncome",
    "AverageChildren",
    "AverageCarsOwned"
).where(
    col("Country").isNull()
).show()

+-------+-------------+-------------------+---------------+----------------+
|Country|CustomerCount|AverageYearlyIncome|AverageChildren|AverageCarsOwned|
+-------+-------------+-------------------+---------------+----------------+
+-------+-------------+-------------------+---------------+----------------+



In [33]:
from pyspark.sql.functions import sum as spark_sum

country_metrics_df.select(
    spark_sum("CustomerCount").alias("TotalCustomers")
).show()

+--------------+
|TotalCustomers|
+--------------+
|         18484|
+--------------+



## 6. Final Analytical Output

Write the country-level customer metrics to Parquet as the final analytical dataset.

The output contains customer counts, average yearly income, average number of children, and average number of cars owned for each country.

In [34]:
country_metrics_df.write \
    .mode("overwrite") \
    .parquet("final_customer_country_metrics")

In [35]:
final_metrics_df = spark.read.parquet("final_customer_country_metrics")

final_metrics_df.show(truncate=False)

+--------------+-------------+-------------------+---------------+----------------+
|Country       |CustomerCount|AverageYearlyIncome|AverageChildren|AverageCarsOwned|
+--------------+-------------+-------------------+---------------+----------------+
|United States |7819         |63616.83           |2.0            |1.52            |
|Australia     |3591         |64338.62           |1.63           |1.91            |
|United Kingdom|1913         |52169.37           |1.63           |1.22            |
|France        |1810         |35762.43           |1.74           |1.25            |
|Germany       |1780         |42943.82           |1.67           |1.21            |
|Canada        |1571         |57167.41           |2.12           |1.46            |
+--------------+-------------+-------------------+---------------+----------------+

